### 什么是节点(Node)
在LangGraph中，节点(Node)是Graph的基本计算单元。  

每个节点：  
- 接收当前State(状态)作为输入
- 执行某些业务逻辑
- 返回一个字典，LangGraph将其合并(reduce)到全局State中

### LangGraph中的节点类型
LangGraph 节点
│  
├── 按实现方式  
│   ├── 函数节点（普通 Python 函数）  
│   ├── 异步函数节点（async def）  
│   ├── 类方法节点（可调用对象）  
│   └── 预置节点（ToolNode 等）  
│  
│  
├── 按控制流角色  
│   ├── 普通节点（顺序执行）  
│   ├── 路由节点（条件分支决策）  
│   ├── 并行节点（同时触发多个分支）  
│   ├── 中断节点（暂停等待外部输入）  
│   └── 终止节点（END）
│   
│   
└── 按组合方式  
    ├── 原子节点（单一职责）  
    ├── 子图节点（内嵌完整 Graph）  
    └── Map-Reduce 节点（Send API 动态扇出）  

In [ ]:
from typing import TypedDict


# 准备工作
# 定义一个State
class StateDemo(TypedDict):
    messages: list[str]

In [ ]:
# 1. 基础节点
"""
基础节点是LangGraph中最基础、最常用的节点类型
满足以下签名规范
"""
def node_demo(state:StateDemo)-> dict:
    """
    :argument
        state: 当前完整的 Graph 状态
    :return
        一个字典，表示对 state 的更新
    """
    new_value = 0
    return {'key': new_value}


In [ ]:
# 2. 可调用对象作为节点
class LLMNode:
    """
    将 LLM 调用封装为一个可调用的节点类
    优点：
      - 可以在__init__中注入依赖(LLM实例，配置等)
      - 支持状态管理(调用次数等)
      - 更好的代码组织
    """
    def __init__(self, llm, system_prompt:str = '你是一个有用的编程助手'):
        self.llm = llm
        self.system_prompt = system_prompt
        self.call_count = 0
    
    def __call__(self, state:StateDemo):
        self.call_count += 1
        return self.call_count
    
# 使用方式：
# llm_node = LLMNode(llm=your_llm, system_prompt=system_prompt)
# graph.add_node('llm', llm_node) # 直接传入实例

In [ ]:
# 3. 条件路由
"""
在LangGraph中，条件路由有两种方式：
    - 路由函数（边上路由， 传统方式）
    - Command节点（节点内路由）
"""
